In [1]:
import ee

ee.Authenticate()
ee.Initialize(project='avilight-483312')

# --- Time range: January 2014 through December 2025 ---
START_DATE = ee.Date('2014-01-01')
N_MONTHS = 144

# --- Google Drive export ---
DRIVE_FOLDER = 'Satellite Datasets'
EXPORT_NAME = 'province_lst_day_monthly_2014_2025'

# --- Philippine province boundaries ---
provinces = (
    ee.FeatureCollection('FAO/GAUL/2025/level2')
    .filter(ee.Filter.eq('GAUL0_NAME', 'Philippines'))
)

print('Province features:', provinces.size().getInfo())


def mask_scale_lst(image):
    """
    Keep MODIS daytime LST pixels whose QA bits 0-1 are 0 or 1,
    then convert Kelvin-scaled integers to degrees Celsius.
    """
    qa_mask = image.select('QC_Day').bitwiseAnd(3).lte(1)

    return (
        image.select('LST_Day_1km')
        .updateMask(qa_mask)
        .multiply(0.02)
        .subtract(273.15)
        .rename('lst_day')
    )


def monthly_lst(month_index):
    """Create a monthly-mean daytime LST image plus valid-pixel band."""
    month_index = ee.Number(month_index)

    d0 = START_DATE.advance(month_index, 'month')
    d1 = d0.advance(1, 'month')

    monthly_collection = (
        ee.ImageCollection('MODIS/061/MOD11A2')
        .filterDate(d0, d1)
        .map(mask_scale_lst)
    )

    # Mean of all quality-approved 8-day LST composites in the month.
    lst_day = monthly_collection.mean().rename('lst_day')

    # Province mean of this band is the usable-pixel share for the month.
    lst_valid = lst_day.mask().rename('lst_valid')

    return (
        lst_day.addBands(lst_valid)
        .set({
            'year': d0.get('year'),
            'month': d0.get('month'),
            'date': d0.format('YYYY-MM')
        })
    )


# Build monthly images for 2014-01 through 2025-12.
month_indices = ee.List.sequence(0, N_MONTHS - 1)

lst_monthly = ee.ImageCollection.fromImages(
    month_indices.map(monthly_lst)
)


def reduce_month_to_provinces(image):
    """Calculate mean daytime LST and valid-pixel share per province."""
    reduced = image.reduceRegions(
        collection=provinces,
        reducer=ee.Reducer.mean(),
        scale=1000,
        tileScale=4
    )

    def format_row(feature):
        return ee.Feature(None, {
            'adm1': feature.get('GAUL1_NAME'),
            'province': feature.get('GAUL2_NAME'),
            'adm2_code': feature.get('GAUL2_CODE'),
            'year': image.get('year'),
            'month': image.get('month'),
            'lst_day': feature.get('lst_day'),
            'lst_valid': feature.get('lst_valid')
        })

    return reduced.map(format_row)


province_lst_fc = lst_monthly.map(
    reduce_month_to_provinces
).flatten()

province_lst_export = province_lst_fc.select([
    'adm1',
    'province',
    'adm2_code',
    'year',
    'month',
    'lst_day',
    'lst_valid'
])

task = ee.batch.Export.table.toDrive(
    collection=province_lst_export,
    description=EXPORT_NAME,
    folder=DRIVE_FOLDER,
    fileNamePrefix=EXPORT_NAME,
    fileFormat='CSV'
)

task.start()

print('✅ Province-level monthly daytime LST export started')
print('Task ID:', task.id)
print('Drive folder:', DRIVE_FOLDER)
print('Period: January 2014 to December 2025')

Province features: 85
✅ Province-level monthly daytime LST export started
Task ID: GDJ75YOXM7DEQMWXRJN6E6X3
Drive folder: Satellite Datasets
Period: January 2014 to December 2025


In [5]:
print(task.status())

{'state': 'COMPLETED', 'description': 'province_lst_day_monthly_2014_2025', 'priority': 100, 'creation_timestamp_ms': 1794716728194, 'update_timestamp_ms': 1794716993235, 'start_timestamp_ms': 1794716733306, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1-bU5d-UWd39u4OkyDb09gBu3DzCrayi9'], 'attempt': 1, 'batch_eecu_usage_seconds': 1673.92138671875, 'id': 'GDJ75YOXM7DEQMWXRJN6E6X3', 'name': 'projects/avilight-483312/operations/GDJ75YOXM7DEQMWXRJN6E6X3'}


In [6]:
import pandas as pd

df_lst = pd.read_csv('province_lst_day_monthly_2014_2025.csv')

print('Total rows:', len(df_lst))
print('Provinces:', df_lst['adm2_code'].nunique())
print('Months:', df_lst[['year', 'month']].drop_duplicates().shape[0])
print('Expected rows:', df_lst['adm2_code'].nunique() * 144)
print('Low-valid rows:', (df_lst['lst_valid'] < 0.3).sum())

df_lst.loc[df_lst['lst_valid'] < 0.3, 'lst_day'] = pd.NA
df_lst.to_parquet('province_lst_day_monthly.parquet', index=False)

Total rows: 12240
Provinces: 85
Months: 144
Expected rows: 12240
Low-valid rows: 36
